<a href="https://colab.research.google.com/github/nexusFour2026/test/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()


Saving dataset-1.csv to dataset-1 (1).csv


In [3]:
import pandas as pd

df = pd.read_csv("dataset-1.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False
3,4922-CVPDX,Female,0,Yes,No,26,Yes,No,DSL,No,...,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.7,False
4,2903-YYTBW,Male,0,Yes,Yes,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,False


In [11]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [12]:
df.duplicated().sum()

np.int64(0)

In [4]:
def data_cleaning(df):
    df = df.copy()

    # ========================================================
    # 1. Remove customerID
    # ========================================================

    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])

    # ========================================================
    # 2. Convert TotalCharges to numeric
    # ========================================================

    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(
            df["TotalCharges"],
            errors="coerce"
        )

    # ========================================================
    # 3. Convert Churn to 0/1
    # ========================================================

    if TARGET_COLUMN in df.columns:

        if df[TARGET_COLUMN].dtype == bool:

            df[TARGET_COLUMN] = (
                df[TARGET_COLUMN]
                .astype(int)
            )

        else:

            df[TARGET_COLUMN] = (
                df[TARGET_COLUMN]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({
                    "true": 1,
                    "false": 0,
                    "1": 1,
                    "0": 0,
                    "yes": 1,
                    "no": 0
                })
            )

    # ========================================================
    # 4. Remove unnecessary spaces from text columns
    # ========================================================

    object_columns = df.select_dtypes(
        include=["object"]
    ).columns

    for column in object_columns:

        df[column] = df[column].apply(
            lambda x: x.strip()
            if isinstance(x, str)
            else x
        )

    # ========================================================
    # IMPORTANT:
    # Do not drop rows here.
    # Missing values will be handled by the Pipeline.
    # ========================================================

    return df

In [13]:
import pandas as pd

df = pd.read_csv("dataset-1.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False
3,4922-CVPDX,Female,0,Yes,No,26,Yes,No,DSL,No,...,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.7,False
4,2903-YYTBW,Male,0,Yes,Yes,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,False


In [6]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):
    df = df.copy()

    # --------------------------------------------------------
    # 1. Convert tenure from months to years
    # --------------------------------------------------------
    if "tenure" in df.columns:
        df["tenure_years"] = df["tenure"] / 12.0

        # ----------------------------------------------------
        # 2. Create customer tenure groups
        # ----------------------------------------------------
        df["tenure_group"] = pd.cut(
            df["tenure"],
            bins=[-1, 6, 12, 24, 48, 72],
            labels=[
                "0-6 months",
                "7-12 months",
                "13-24 months",
                "25-48 months",
                "49-72 months"
            ]
        )

    # --------------------------------------------------------
    # 3. Count the number of services used by the customer
    # --------------------------------------------------------
    service_columns = [
        "PhoneService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    existing_service_columns = [
        col for col in service_columns
        if col in df.columns
    ]

    if existing_service_columns:

        df["num_services"] = (
            df[existing_service_columns]
            .eq("Yes")
            .sum(axis=1)
        )

    # --------------------------------------------------------
    # 4. Average monthly spending
    # --------------------------------------------------------
    if "TotalCharges" in df.columns and "tenure" in df.columns:

        tenure_for_calculation = df["tenure"].replace(0, np.nan)

        df["avg_monthly_spend"] = (
            df["TotalCharges"] / tenure_for_calculation
        )

    return df

In [7]:
# ============================================================
# SCALE FEATURES
# ============================================================

def scale_features(df):
    # Scaling and encoding will be performed inside
    # the scikit-learn Pipeline.
    return df.copy()

In [8]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

def split_data(df):

    # --------------------------------------------------------
    # First split:
    # 60% training
    # 40% temporary data
    # --------------------------------------------------------
    train_df, temp_df = train_test_split(
        df,
        test_size=0.40,
        random_state=RANDOM_STATE,
        stratify=df[TARGET_COLUMN]
    )

    # --------------------------------------------------------
    # Second split:
    # Half of temporary data = validation
    # Half of temporary data = test
    #
    # Therefore:
    # Train      = 60%
    # Validation = 20%
    # Test       = 20%
    # --------------------------------------------------------
    validation_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp_df[TARGET_COLUMN]
    )

    return train_df, validation_df, test_df

In [10]:
# ============================================================
# SPLIT RAW DATA
# ============================================================

train_df, validation_df, test_df = split_data(df)

print("Training set:", train_df.shape)
print("Validation set:", validation_df.shape)
print("Test set:", test_df.shape)


# ============================================================
# PREPROCESS EACH DATA SPLIT
# ============================================================

train_processed = preprocess_data(train_df)
validation_processed = preprocess_data(validation_df)
test_processed = preprocess_data(test_df)


# ============================================================
# SEPARATE FEATURES AND TARGET
# ============================================================

X_train = train_processed.drop(columns=[TARGET_COLUMN])
y_train = train_processed[TARGET_COLUMN]

X_validation = validation_processed.drop(columns=[TARGET_COLUMN])
y_validation = validation_processed[TARGET_COLUMN]

X_test = test_processed.drop(columns=[TARGET_COLUMN])
y_test = test_processed[TARGET_COLUMN]


print("\nX_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)


# ============================================================
# IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


# ============================================================
# NUMERICAL PIPELINE
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# CATEGORICAL PIPELINE
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)


# ============================================================
# COMPLETE PREPROCESSING PIPELINE
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ============================================================
# DIFFERENT CLASSIFICATION MODELS
# ============================================================

models = {

    "Logistic Regression C=0.1":
        LogisticRegression(
            C=0.1,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

    "Logistic Regression C=1.0":
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

    "Logistic Regression C=2.0":
        LogisticRegression(
            C=2.0,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
}


# ============================================================
# THRESHOLDS FOR F1 OPTIMIZATION
# ============================================================

thresholds = np.arange(
    0.20,
    0.71,
    0.01
)


# ============================================================
# TRAIN MODELS
# ============================================================

results = []

trained_models = {}

best_thresholds = {}


for model_name, classifier in models.items():

    print("\n" + "=" * 70)
    print("Training:", model_name)
    print("=" * 70)

    # --------------------------------------------------------
    # Create ONE complete pipeline
    # preprocessing + model
    # --------------------------------------------------------
    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                classifier
            )
        ]
    )

    # --------------------------------------------------------
    # Fit ONLY on training data
    # --------------------------------------------------------
    pipeline.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Get validation probabilities
    # --------------------------------------------------------
    validation_probabilities = (
        pipeline.predict_proba(X_validation)[:, 1]
    )

    # --------------------------------------------------------
    # Find threshold that gives highest validation F1
    # --------------------------------------------------------
    best_threshold = 0.50
    best_validation_f1 = -1

    for threshold in thresholds:

        validation_predictions = (
            validation_probabilities >= threshold
        ).astype(int)

        current_f1 = f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        )

        if current_f1 > best_validation_f1:

            best_validation_f1 = current_f1
            best_threshold = threshold

    # --------------------------------------------------------
    # Training predictions
    # --------------------------------------------------------
    train_probabilities = (
        pipeline.predict_proba(X_train)[:, 1]
    )

    train_predictions = (
        train_probabilities >= best_threshold
    ).astype(int)

    # --------------------------------------------------------
    # Validation predictions
    # --------------------------------------------------------
    validation_predictions = (
        validation_probabilities >= best_threshold
    ).astype(int)

    # --------------------------------------------------------
    # Training metrics
    # --------------------------------------------------------
    train_accuracy = accuracy_score(
        y_train,
        train_predictions
    )

    train_precision = precision_score(
        y_train,
        train_predictions,
        zero_division=0
    )

    train_recall = recall_score(
        y_train,
        train_predictions,
        zero_division=0
    )

    train_f1 = f1_score(
        y_train,
        train_predictions,
        zero_division=0
    )

    train_auc = roc_auc_score(
        y_train,
        train_probabilities
    )

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------
    validation_accuracy = accuracy_score(
        y_validation,
        validation_predictions
    )

    validation_precision = precision_score(
        y_validation,
        validation_predictions,
        zero_division=0
    )

    validation_recall = recall_score(
        y_validation,
        validation_predictions,
        zero_division=0
    )

    validation_f1 = f1_score(
        y_validation,
        validation_predictions,
        zero_division=0
    )

    validation_auc = roc_auc_score(
        y_validation,
        validation_probabilities
    )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------
    results.append({

        "model": model_name,

        "threshold": best_threshold,

        "train_accuracy": train_accuracy,
        "train_precision": train_precision,
        "train_recall": train_recall,
        "train_f1": train_f1,
        "train_roc_auc": train_auc,

        "validation_accuracy": validation_accuracy,
        "validation_precision": validation_precision,
        "validation_recall": validation_recall,
        "validation_f1": validation_f1,
        "validation_roc_auc": validation_auc
    })

    trained_models[model_name] = pipeline
    best_thresholds[model_name] = best_threshold


# ============================================================
# DISPLAY MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="validation_f1",
    ascending=False
).reset_index(drop=True)

display(results_df)

NameError: name 'train_test_split' is not defined